# Real-Time Face Mask Detection Using YOLOv5


### 0. Random Seeds
Seeds are set for Python, NumPy, and PyTorch to reduce variation. Exact reproducibility also depends on the dataset, software versions, hardware, and execution order.

Run this notebook from top to bottom in a fresh Google Colab runtime. Saved outputs from earlier sessions have been cleared because they referenced different training runs. This publication copy has not been rerun; evaluate it before reporting numerical results.


In [ ]:
import random
import numpy as np
import torch
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Force deterministic algorithms in CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set seed for YOLOv5 training calls as well
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Global random seed set to: {seed}")

set_seed(42)

### 1. Environment Configuration and API Authentication
This section initializes the environment and configures the Kaggle API credentials required to securely access and download the target dataset.

In [ ]:
!pip install -q kaggle
import os

# Create the necessary directory
os.makedirs('/root/.kaggle', exist_ok=True)

# Check if file exists and move it
# Please upload your kaggle.json file to the Colab file browser before running this
if os.path.exists('/content/kaggle.json'):
    !cp /content/kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json
    print("Kaggle API key successfully configured!")
else:
    print("Error: 'kaggle.json' not found in /content/. Please upload it using the file sidebar.")

### 2. Dataset Acquisition and Extraction
The Face Mask Detection dataset is retrieved directly from the Kaggle repository and extracted into the local environment for subsequent processing and conversion.

In [ ]:
!kaggle datasets download -d andrewmvd/face-mask-detection
!unzip -o face-mask-detection.zip -d /content/archive
print("Dataset downloaded and extracted to /content/archive")

### 3. Annotation Preprocessing
*   **Conversion**: Pascal VOC (XML) to YOLO (.txt) format.
*   **Normalization**: Scaling bounding box coordinates to the $[0, 1]$ range.
*   **Data Partitioning**: Deterministic 80/20 split for training and validation.

In [ ]:
import os
import glob
import xml.etree.ElementTree as ET
import random
import shutil
from pathlib import Path

# --- Configuration ---
DATASET_PATH = "/content/archive"
IMAGE_DIR = os.path.join(DATASET_PATH, "images")
ANNOTATION_DIR = os.path.join(DATASET_PATH, "annotations")
OUTPUT_DIR = "/content/yolo_dataset"

CLASSES = ["with_mask", "without_mask", "mask_weared_incorrect"]

def convert_to_yolo_bbox(size, box):
    """Converts VOC bounding box to YOLO normalized format."""
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x_center = (box[0] + box[1]) / 2.0
    y_center = (box[2] + box[3]) / 2.0
    width = box[1] - box[0]
    height = box[3] - box[2]
    return (x_center * dw, y_center * dh, width * dw, height * dh)

def parse_xml_and_export(xml_file, output_txt_path):
    """Parses a single XML and writes the YOLO formatted text file."""
    tree = ET.parse(xml_file)
    root = tree.getroot()

    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    with open(output_txt_path, 'w') as f:
        for obj in root.iter('object'):
            cls = obj.find('name').text
            if cls not in CLASSES:
                continue
            cls_id = CLASSES.index(cls)
            xmlbox = obj.find('bndbox')
            b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                 float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
            bb = convert_to_yolo_bbox((w, h), b)
            f.write(f"{cls_id} {' '.join([str(a) for a in bb])}\n")

# --- Create Split Directories ---
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', split), exist_ok=True)

# --- Process and Split Data ---
print("Locating images and converting annotations...")
all_images = glob.glob(os.path.join(IMAGE_DIR, "*.png"))

# CRITICAL FOR REPRODUCIBILITY: Sort before shuffling
all_images.sort()
random.seed(42)
random.shuffle(all_images)

split_ratio = 0.8
train_count = int(len(all_images) * split_ratio)

for i, img_path in enumerate(all_images):
    filename = Path(img_path).stem
    xml_path = os.path.join(ANNOTATION_DIR, f"{filename}.xml")

    if not os.path.exists(xml_path):
        continue

    split = 'train' if i < train_count else 'val'

    # Copy image
    dest_img_path = os.path.join(OUTPUT_DIR, 'images', split, f"{filename}.png")
    shutil.copy(img_path, dest_img_path)

    # Process and write label
    dest_label_path = os.path.join(OUTPUT_DIR, 'labels', split, f"{filename}.txt")
    parse_xml_and_export(xml_path, dest_label_path)

print(f"Dataset successfully converted! Train: {train_count}, Val: {len(all_images) - train_count}")

### 4. Model Configuration
*   **Framework**: YOLOv5 repository setup and environment initialization.
*   **Data Definition**: Mapping dataset paths and class labels via `mask_data.yaml`.
*   **Hyperparameter Tuning**: Customizing augmentations (Mosaic, MixUp) for robust feature extraction.

In [ ]:
# Set up YOLOv5 from a consistent working directory.
%cd /content
!git clone https://github.com/ultralytics/yolov5
%cd /content/yolov5
!pip install -qr requirements.txt

# Use the same run for training, evaluation, video detection, and benchmarking.
from pathlib import Path
RUN_NAME = "face_mask_det"
RUN_DIR = Path("/content/yolov5/runs/train") / RUN_NAME
if RUN_DIR.exists():
    raise FileExistsError("This run already exists. Choose a new RUN_NAME and rerun this cell before training.")


In [ ]:
%%writefile mask_data.yaml
path: /content/yolo_dataset  # Absolute path to your dataset
train: images/train
val: images/val

# Classes
nc: 3
names: ['with_mask', 'without_mask', 'mask_weared_incorrect']

In [ ]:
%%writefile hyp.custom.yaml
# Inherit standard hyperparams and boost augmentations
lr0: 0.01   # learning rate. Controls how big a correction the model makes each time it's wrong.

# the learning rate isn't constant; it starts low, ramps up (warmup), then gradually decays
# toward lr0 × lrf by the end. This avoids destabilizing the pretrained weights early on.
lrf: 0.1
# technical knobs for the optimizer (the algorithm doing the adjusting) that help it converge smoothly and avoid overfitting.
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 3.0
warmup_momentum: 0.8
warmup_bias_lr: 0.1
box: 0.05
cls: 0.5
cls_pw: 1.0
obj: 1.0
obj_pw: 1.0
iou_t: 0.20
anchor_t: 4.0
fl_gamma: 0.0
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.5
mosaic: 1.0       # 100% probability of Mosaic
mixup: 0.3        # 30% probability of MixUp
copy_paste: 0.0

### 5. Training Process
*   **Transfer Learning**: Utilizing pre-trained weights from COCO (Common Objects in Context) for accelerated convergence.
*   **Optimization**: 50 epochs using Stochastic Gradient Descent (SGD) with a 0.01 learning rate.
*   **Performance Metrics**: Monitoring box, objectness, and classification loss components.

In [ ]:
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data mask_data.yaml \
    --hyp hyp.custom.yaml \
    --weights yolov5s.pt \
    --project /content/yolov5/runs/train \
    --name {RUN_NAME} \
    --cache \
    --seed 42


### 6. Performance Evaluation
Use the validation output from the training run above to report precision, recall, mAP@0.5, and mAP@0.5:0.95. The split is training/validation only; this notebook does not establish performance on an independent held-out test set.

Previous saved results referred to different runs and did not match the written summary, so no numerical performance claim is retained in this publication copy.


### Metric Definition: mean Average Precision (mAP)
*   **Precision**: Accuracy of positive predictions (True Positives / Total Predicted Positives).
*   **Recall**: Ability to find all positive instances (True Positives / Total Actual Positives).
*   **mAP@.5**: The mean of Average Precision calculated at an Intersection over Union (IoU) threshold of 0.50. It represents the area under the Precision-Recall curve, serving as a comprehensive measure of the model's localization and classification performance.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RUN_DIR / 'PR_curve.png'), width=800))


### 6.1 Class Confusion Analysis
The confusion matrix helps identify specific classification errors, such as whether the model is confusing the 'incorrectly worn' class with 'with mask'. This is particularly important for imbalanced datasets.

In [ ]:
confusion_matrix_path = RUN_DIR / 'confusion_matrix.png'
if confusion_matrix_path.exists():
    display(Image(filename=str(confusion_matrix_path), width=800))
else:
    print(f"Confusion matrix not found: {confusion_matrix_path}. Complete training first.")


In [ ]:
!wget -O /content/test_video.mp4 "https://github.com/intel-iot-devkit/sample-videos/raw/master/head-pose-face-detection-female-and-male.mp4"

### 6.1 Real-Time Inference Pipeline
*   **Execution**: Utilizing `detect.py` to run the optimized `best.pt` weights on unseen video data.
*   **Thresholding**: Applied a confidence threshold of 0.40 to filter low-probability detections.
*   **Output**: Generation of an annotated video stream with bounding boxes and classification labels.

In [ ]:
!python detect.py \
    --weights {RUN_DIR / 'weights' / 'best.pt'} \
    --source /content/test_video.mp4 \
    --conf 0.40


In [ ]:
# Environment Fix: Removing the global 'utils' conflict and clearing the module cache
!pip uninstall -y utils

import sys
import os

# 1. Clear the cached 'utils' module so Python re-imports from the local folder
if 'utils' in sys.modules:
    del sys.modules['utils']

# 2. Force Python to look at the local yolov5 folder first
local_yolov5 = '/content/yolov5'

print("Memory cache cleared and conflict resolved. You can now run the benchmark cell.")

### 6.2 Exploratory Speed Comparison
This cell compares the trained YOLOv5 model with a default pretrained Faster R-CNN model on one validation image. Faster R-CNN is not trained for this project's mask classes.

This is an exploratory timing script, not a controlled performance benchmark: model preprocessing differs and GPU timing is not explicitly synchronized. Do not use its speed ratio as a validated project result.


In [ ]:
import torch
import time
import cv2
import glob
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# --- Configuration ---
YOLO_WEIGHTS = str(RUN_DIR / 'weights' / 'best.pt')
# Grab the first image from your validation set to use for the speed test
TEST_IMAGE = glob.glob('/content/yolo_dataset/images/val/*.png')[0]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ITERATIONS = 100 # Run 100 times to get a fair average speed

def benchmark_yolov5():
    print("Loading YOLOv5...")
    model = torch.hub.load('/content/yolov5', 'custom', path=YOLO_WEIGHTS, source='local', force_reload=True).to(DEVICE)
    model.eval()

    img = cv2.imread(TEST_IMAGE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Warmup the GPU
    for _ in range(10):
        _ = model(img_rgb)

    print("Benchmarking YOLOv5...")
    start_time = time.time()
    for _ in range(ITERATIONS):
        _ = model(img_rgb)
    end_time = time.time()

    fps = ITERATIONS / (end_time - start_time)
    return fps

def benchmark_faster_rcnn():
    print("\nLoading Faster R-CNN (Baseline)...")
    # Load PyTorch's default Faster R-CNN
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT").to(DEVICE)
    model.eval()

    img = cv2.imread(TEST_IMAGE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_tensor = torchvision.transforms.functional.to_tensor(img_rgb).unsqueeze(0).to(DEVICE)

    # Warmup the GPU
    with torch.no_grad():
        for _ in range(10):
            _ = model(img_tensor)

    print("Benchmarking Faster R-CNN...")
    start_time = time.time()
    with torch.no_grad():
        for _ in range(ITERATIONS):
            _ = model(img_tensor)
    end_time = time.time()

    fps = ITERATIONS / (end_time - start_time)
    return fps

# --- Run Benchmarks ---
print(f"Running speed tests on: {DEVICE}\n")
yolo_fps = benchmark_yolov5()
frcnn_fps = benchmark_faster_rcnn()

# --- Print Final Comparison Table ---
print("\n" + "="*45)
print("          BENCHMARK COMPARISON TABLE")
print("="*45)
print(f"{'Model':<15} | {'Speed (FPS)':<15}")
print("-" * 45)
print(f"{'YOLOv5':<15} | {yolo_fps:>10.2f} FPS")
print(f"{'Faster R-CNN':<15} | {frcnn_fps:>10.2f} FPS")
print("-" * 45)
print(f"Exploratory timing only: YOLOv5 measured {yolo_fps / frcnn_fps:.1f}x faster!")
print("="*45)

### 7. Summary
This notebook covers dataset conversion, YOLOv5s training, validation visualizations, video inference, and an exploratory speed comparison.

Deployment performance on a car or other embedded hardware requires separate measurements on that device. Colab GPU results alone do not establish embedded-device performance.
